# Notebook 4: Expert Bucketed Clustering Engine
---
## Strategy
A **hierarchical divide-and-conquer** approach:
1. Pre-classify titles into semantic buckets (Geopolitics, Elections, Safety, etc.)
2. Run independent BERTopic within each bucket for granular sub-topics
3. Label each sub-topic using a local LLM (Ollama/Qwen) on the fly
4. Produce a nested JSON hierarchy

## Flow
```
Raw Data -> Coarse Bucket Assignment (regex rules)
         -> Per-Bucket BERTopic (min_cluster=18)
         -> LLM labels each sub-topic immediately
         -> Nested JSON output
```

## When to Use
- When you have KNOWN high-level categories
- For news dashboards with nested navigation
- When you want to process very large datasets efficiently

## Key Insight
- Each bucket gets its own UMAP/HDBSCAN space
- This prevents a dominant topic (Iran War) from drowning smaller ones


In [ ]:
!pip install -q bertopic sentence-transformers hdbscan umap-learn pandas scikit-learn requests
print("Ready")

In [ ]:
import pandas as pd, numpy as np, re, html, json, requests
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

## Bucket Definitions
These regex rules pre-classify titles into high-level domains.

In [ ]:
BUCKETS = {
    "geopolitical": r"iran|israel|trump|war|military|army|navy|attack|missile|ceasefire",
    "elections": r"election|vote|ballot|bjp|congress|candidate|campaign|poll",
    "politicians": r"modi|shah|gandhi|mamata|yogi|kejriwal|kharge|rahul",
    "safety": r"accident|fire|flood|earthquake|rescue|police|crime|murder",
    "economy": r"petrol|diesel|lpg|price|tax|budget|rupee|inflation",
    "culture": r"bollywood|cricket|ipl|movie|song|festival|temple",
}

def assign_bucket(text):
    for name, pattern in BUCKETS.items():
        if re.search(pattern, text, re.I): return name
    return "general"

DOMAIN_STOPS = {"abp","shorts","viral","subscribe","youtube","news","live","at2","n18v"}
def clean(text):
    if not isinstance(text, str): return ""
    text = html.unescape(text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"#(\w+)", r" \1 ", text)
    text = re.sub(r"[^\w\s\u0900-\u097F]", " ", text)
    tokens = [t for t in text.lower().split() if not (t.isascii() and len(t)<=2) and t not in DOMAIN_STOPS]
    return " ".join(tokens).strip()

# --- UNIVERSAL DATA LOADER ---
import pandas as pd
import os

# Get the file extension
file_ext = os.path.splitext(filename)[1].lower()

if file_ext == '.xlsx':
    print("📂 Detected Excel file. Using read_excel...")
    df = pd.read_excel(filename)
elif file_ext == '.csv' or file_ext == '.txt':
    print("📂 Detected Text/CSV file. Using read_csv with encoding recovery...")
    try:
        df = pd.read_csv(filename, encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(filename, encoding='latin1')
        except:
            df = pd.read_csv(filename, sep="\v", engine="python", encoding='latin1')
else:
    print(f"⚠️ Unknown format {file_ext}. Attempting generic read...")
    df = pd.read_csv(filename, sep="\v", engine="python", encoding='latin1')

# Ensure we have a title column
if "title" not in df.columns:
    if len(df.columns) == 1:
        df.columns = ["title"]
    else:
        df.columns = ["title"] + list(df.columns[1:])
# ------------------------------
if "title" not in df.columns: df.columns=["title"]+list(df.columns[1:])
df["clean_text"] = df["title"].fillna("").astype(str).map(clean)
df = df[df["clean_text"]!=""].reset_index(drop=True)
df["bucket"] = df["clean_text"].map(assign_bucket)
print(df["bucket"].value_counts())

## Per-Bucket Clustering

In [ ]:
embedder = SentenceTransformer("sentence-transformers/LaBSE")
embeddings = embedder.encode(df["clean_text"].tolist(), show_progress_bar=True, batch_size=32)

hierarchy = {}
df["topic_id"] = -1
gid = 0

for bname, bdf in df.groupby("bucket"):
    if len(bdf) < 15:
        print(f"Skipping {bname} ({len(bdf)} docs)")
        continue
    idx = bdf.index.tolist()
    m = BERTopic(embedding_model=embedder, min_topic_size=18, verbose=False)
    t, _ = m.fit_transform(bdf["clean_text"].tolist(), embeddings[idx])
    
    unique_t = sorted(set(ti for ti in t if ti != -1))
    sub_topics = []
    for local_id in unique_t:
        global_id = gid
        gid += 1
        mask = np.array(t) == local_id
        for i, m_val in zip(idx, mask):
            if m_val: df.loc[i, "topic_id"] = global_id
        kws = [w for w,_ in m.get_topic(local_id)][:5]
        sub_topics.append({"id": global_id, "keywords": kws, "count": int(mask.sum())})
    
    hierarchy[bname] = {"total": len(bdf), "sub_topics": sub_topics}
    print(f"{bname}: {len(unique_t)} sub-topics from {len(bdf)} docs")

noise_pct = round(100*(df["topic_id"]==-1).mean(),1)
print(f"\nTotal: {gid} topics, {noise_pct}% noise")

## Nested Hierarchy Output

In [ ]:
print(json.dumps(hierarchy, indent=2, ensure_ascii=False)[:3000])

In [ ]:
## FINAL CLEAN STANDARDIZED EXPORT
# 1. Create a CLEAN Topic Label Map (No numbers, Space-separated, Title-case)
topic_info = model.get_topic_info()
label_map = {}
for _, row in topic_info.iterrows():
    t_id = row['Topic']
    raw_name = row['Name']
    # Remove the "0_" or "-1_" prefix and clean up underscores
    if "_" in raw_name:
        clean_name = raw_name.split("_", 1)[-1].replace("_", " ").title()
    else:
        clean_name = raw_name.title()
    label_map[t_id] = clean_name

df['topic_label'] = df['topic_id'].map(label_map)

# 2. Reorganize Columns
if 'ner_text' not in df.columns: df['ner_text'] = df['clean_text']
final_df = df.rename(columns={'title': 'raw_text'})

# 3. Selection (Exactly 5 columns)
cols_to_keep = ['raw_text', 'clean_text', 'ner_text', 'topic_id', 'topic_label']
final_df = final_df[cols_to_keep]

# 4. Save and Download
output_fn = "final_clustering_results.csv"
final_df.to_csv(output_fn, index=False)
print(f"Success! Exported {len(final_df)} titles with clean, unique labels.")
from google.colab import files
files.download(output_fn)